# 北上广深租房市场数据分析

## 第二部分：数据字典与数据质量深入检查

数据来源：Alfred1984/interesting-python 项目的 BSGS_Rent 样本数据。

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/data_sample.csv")
df = pd.read_csv(DATA_PATH)

print("数据规模：", df.shape)

数据规模： (12000, 20)


## 一、建立数据字典

In [4]:
print("字段数量：", len(df.columns))
df.columns.tolist()

字段数量： 20


['_id',
 'bathroom_num',
 'bedroom_num',
 'bizcircle_name',
 'city',
 'dist',
 'distance',
 'frame_orientation',
 'hall_num',
 'house_tag',
 'house_title',
 'latitude',
 'layout',
 'longitude',
 'm_url',
 'rent_area',
 'rent_price_listing',
 'rent_price_unit',
 'resblock_name',
 'type']

In [5]:
field_meanings = {
    "_id": "MongoDB自动生成的记录ID",
    "bathroom_num": "卫生间数量",
    "bedroom_num": "卧室数量",
    "bizcircle_name": "商圈名称",
    "city": "城市名称",
    "dist": "行政区名称",
    "distance": "距离最近地铁站的距离（米）",
    "frame_orientation": "房屋朝向",
    "hall_num": "客厅数量",
    "house_tag": "房源标签",
    "house_title": "房源标题",
    "latitude": "纬度",
    "layout": "户型描述",
    "longitude": "经度",
    "m_url": "链家移动端房源链接",
    "rent_area": "出租面积（平方米）",
    "rent_price_listing": "挂牌月租金",
    "rent_price_unit": "租金单位",
    "resblock_name": "小区名称",
    "type": "出租类型"
}

In [6]:
# 把字段含义、类型、缺失情况、唯一值数量和示例值组合成一张20行的数据字典
data_dictionary = pd.DataFrame({
    "字段名": df.columns,
    "中文含义": [field_meanings[column] for column in df.columns],
    "数据类型": df.dtypes.astype(str).values,
    "非空数量": df.notna().sum().values,
    "缺失数量": df.isna().sum().values,
    "缺失比例(%)": (df.isna().mean() * 100).round(2).values,
    "唯一值数量": df.nunique(dropna=True).values,
    "示例值": [
        df[column].dropna().iloc[0]
        if df[column].notna().any()
        else np.nan
        for column in df.columns
    ]
})

data_dictionary

,字段名,中文含义,数据类型,非空数量,缺失数量,缺失比例(%),唯一值数量,示例值
0,_id,MongoDB自动生成的记录ID,str,12000,0,0.00,12000,5c714363397be4c5251a3ded
1,bathroom_num,卫生间数量,int64,12000,0,0.00,10,2
2,bedroom_num,卧室数量,int64,12000,0,0.00,13,3
3,bizcircle_name,商圈名称,str,11999,1,0.01,682,上地
4,city,城市名称,str,12000,0,0.00,4,北京
5,dist,行政区名称,str,12000,0,0.00,49,海淀
6,distance,距离最近地铁站的距离（米）,float64,6794,5206,43.38,1154,788.0
7,frame_orientation,房屋朝向,str,11899,101,0.84,96,南 北
8,hall_num,客厅数量,int64,12000,0,0.00,6,2
9,house_tag,房源标签,str,10124,1876,15.63,237,精装 集中供暖 双卫生间


## 二、分类字段取值检查

In [7]:
# 统计：每个城市的房源数量,整租与合租的数量,租金单位是否统一
category_columns = ["city", "type", "rent_price_unit"]

for column in category_columns:
    print(f"\n字段：{column}")
    display(
        df[column]
        .value_counts(dropna=False)
        .rename_axis("取值")
        .reset_index(name="数量")
    )


字段：city


,取值,数量
0,北京,3000
1,上海,3000
2,广州,3000
3,深圳,3000



字段：type


,取值,数量
0,整租,11586
1,合租,414



字段：rent_price_unit


,取值,数量
0,元/月,12000


In [11]:
# 检查四个城市的行政区分布,按城市和行政区分组,统计每个行政区的房源数量,在每个城市中按数量从高到低排列
district_counts = df.groupby(["city", "dist"]).size()

district_counts = district_counts.reset_index()
district_counts.columns = ["city", "dist", "房源数量"]

district_counts = district_counts.sort_values(
    by=["city", "房源数量"],
    ascending=[True, False]
)

district_counts = district_counts.reset_index(drop=True)

district_counts

,city,dist,房源数量
0,上海,浦东,794
1,上海,闵行,287
2,上海,松江,263
3,上海,徐汇,215
4,上海,普陀,192
5,上海,嘉定,189
6,上海,长宁,180
7,上海,青浦,173
8,上海,宝山,167
9,上海,黄浦,111


In [12]:
# 检查最常见的户型
layout_counts = df["layout"].value_counts()

layout_counts.head(10)

layout
2室1厅1卫    2595
1室1厅1卫    1964
1室0厅1卫    1305
3室2厅2卫    1131
2室2厅1卫     910
3室1厅1卫     732
3室2厅1卫     686
4室2厅2卫     369
3室1厅2卫     270
2室2厅2卫     209
Name: count, dtype: int64

In [13]:
# 检查最常见的房屋朝向
orientation_counts = df["frame_orientation"].value_counts()

orientation_counts.head(10)

frame_orientation
南      5235
南 北    1822
东南     1417
北       736
东       696
西南      443
西       336
东北      215
西北      200
东 西     127
Name: count, dtype: int64

In [14]:
# 检查最常见的房源标签，并包含缺失值
house_tag_counts = df["house_tag"].value_counts(dropna=False)  # dropna=False表示不要忽略缺失值，这样可以看到没有标签的房源数量

house_tag_counts.head(10)

house_tag
NaN              1876
近地铁              1834
近地铁 随时看房          719
随时看房              685
近地铁 新上            531
新上                369
独栋公寓              342
独栋公寓 近地铁          325
近地铁 集中供暖 随时看房     317
集中供暖 随时看房         241
Name: count, dtype: int64

In [15]:
# 检查房源数量最多的商圈
bizcircle_counts = df["bizcircle_name"].value_counts(dropna=False)

bizcircle_counts.head(10)

bizcircle_name
西乡       182
龙华中心     175
龙岗中心城    141
坂田       120
科学城      104
福永       100
蛇口        97
大石        91
深圳北站      89
新安        84
Name: count, dtype: int64

## 三、房间数量与异常值检查

In [16]:
# 先看大多数房源是什么样，再找出特别小或特别大的记录
room_columns = ["bedroom_num", "hall_num", "bathroom_num"]

df[room_columns].describe()

,bedroom_num,hall_num,bathroom_num
count,12000.000000,12000.000000,12000.000000
mean,2.242667,1.228167,1.332583
std,1.161849,0.672169,0.682161
min,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000
50%,2.000000,1.000000,1.000000
75%,3.000000,2.000000,2.000000
max,20.000000,5.000000,9.000000


In [17]:
# 查看每种卧室数量分别有多少套房源
bedroom_counts = df["bedroom_num"].value_counts()

bedroom_counts = bedroom_counts.sort_index()

bedroom_counts

bedroom_num
0        3
1     3598
2     4008
3     2950
4      986
5      332
6       79
7       22
8        9
9        9
10       1
13       1
20       2
Name: count, dtype: int64

In [18]:
# 查看这3条“0卧室”房源的具体信息
zero_bedroom = df[df["bedroom_num"] == 0]

zero_bedroom[
    [
        "city",
        "type",
        "bedroom_num",
        "layout",
        "rent_area",
        "rent_price_listing",
        "house_title"
    ]
]
# 这3条记录不是正常的“0卧室”房源：
# layout 显示“未知室”
# 两条标题使用了“--室--厅”
# 无法确认实际卧室数量
# 面积和租金仍然存在，所以不应直接删除整条记录

,city,type,bedroom_num,layout,rent_area,rent_price_listing,house_title
6773,广州,整租,0,未知室0厅0卫,80,4800,东凌广场 --室--厅 4800元
6834,广州,整租,0,未知室1厅1卫,36,3500,侨英花园 0室1厅 3500元
7761,广州,整租,0,未知室0厅0卫,70,4200,东凌广场 --室--厅 4200元


In [19]:
# 检查卧室数量特别大的房源
large_bedroom = df[df["bedroom_num"] >= 10]

large_bedroom[
    [
        "city",
        "type",
        "bedroom_num",
        "layout",
        "rent_area",
        "rent_price_listing",
        "house_title"
    ]
]
# 这4条高卧室数记录不能直接判定为错误：
# 两条广州 20室 和一条深圳 13室 都是合租。
# 合租记录中的 rent_area 很可能是当前出租房间的面积，而 bedroom_num 是整套合租房源的卧室数。
# 深圳 10室 是整租，面积333平方米、租金25,000元，结合标题看像大型别墅，数值基本合理。

,city,type,bedroom_num,layout,rent_area,rent_price_listing,house_title
6152,广州,合租,20,20室2厅8卫,15,2500,合租 · 珠江广场 20室2厅 复式
7540,广州,合租,20,20室2厅8卫,10,1300,合租 · 珠江广场 20室2厅 复式
9647,深圳,合租,13,13室1厅3卫,15,1480,合租 · 振业城 13室1厅
10814,深圳,整租,10,10室3厅7卫,333,25000,龙泉别墅 10室3厅 25000元


In [20]:
# 统计每种客厅数量
hall_counts = df["hall_num"].value_counts()

hall_counts = hall_counts.sort_index()

hall_counts
# 1个客厅最常见：6,391条
# 2个客厅：3,991条
# 0个客厅：1,502条，可能包括开间和合租单间
# 5个客厅只有2条，属于极少数记录，需要单独检查

hall_num
0    1502
1    6391
2    3991
3     101
4      13
5       2
Name: count, dtype: int64

In [21]:
# 查看客厅数量最大的房源
large_hall = df[df["hall_num"] == 5]

large_hall[
    [
        "city",
        "type",
        "bedroom_num",
        "hall_num",
        "layout",
        "rent_area",
        "rent_price_listing",
        "house_title"
    ]
]
# 这两条“5个客厅”的记录是合理的大户型：
# 北京：9室5厅9卫，544平方米，月租40,000元
# 上海：9室5厅5卫，436平方米，月租80,000元
# 户型、面积和租金能够相互对应，很可能是大型别墅。因此：
# hall_num = 5 → 保留，不作为错误值删除

,city,type,bedroom_num,hall_num,layout,rent_area,rent_price_listing,house_title
1206,北京,整租,9,5,9室5厅9卫,544,40000,整租 · 精装修独栋别墅，家具家电齐全，可直接入住！
4680,上海,整租,9,5,9室5厅5卫,436,80000,汤臣高尔夫独栋别墅|大花园 露台| 房间设施配套齐 复式


In [22]:
# 检查“0个客厅”的房源主要是整租还是合租
zero_hall = df[df["hall_num"] == 0]

zero_hall_type_counts = zero_hall["type"].value_counts()

zero_hall_type_counts
# 0客厅的整租房源：1,488条
# 0客厅的合租房源：14条
# 因此，0客厅并不只是合租单间，还可能是开间或“1室0厅”等整租户型。现在还不能把它当作异常值。

type
整租    1488
合租      14
Name: count, dtype: int64

In [23]:
# 查看0客厅房源最常见的户型
zero_hall_layout_counts = zero_hall["layout"].value_counts()

zero_hall_layout_counts.head(10)
# 1,305条是 1室0厅1卫
# 这是常见的开间或无独立客厅户型
# 其他记录也明确写了 0厅
# 因此：hall_num = 0 → 合理值，保留

layout
1室0厅1卫    1305
1房间1卫       71
1室0厅0卫      46
2室0厅1卫      43
1房间0卫        7
3室0厅1卫       3
3室0厅0卫       2
2房间1卫        2
4室0厅1卫       2
6室0厅1卫       2
Name: count, dtype: int64

In [24]:
# 统计每种卫生间数量
bathroom_counts = df["bathroom_num"].value_counts()

bathroom_counts = bathroom_counts.sort_index()

bathroom_counts
# 1个卫生间最常见：8,750条
# 2个卫生间：2,537条
# 6至9个卫生间很少，可能是别墅或大型合租房
# 0个卫生间有87条，需要重点检查

bathroom_num
0      87
1    8750
2    2537
3     440
4     124
5      39
6      11
7       7
8       4
9       1
Name: count, dtype: int64

In [25]:
# 先检查0卫生间房源的出租类型
zero_bathroom = df[df["bathroom_num"] == 0]

zero_bathroom_type_counts = zero_bathroom["type"].value_counts()

zero_bathroom_type_counts
# 0卫生间房源几乎都是整租：\
# 正常整租房源通常应有卫生间，因此这些记录可能是户型信息不完整，而不是真正没有卫生间。

type
整租    86
合租     1
Name: count, dtype: int64

In [26]:
# 查看这些房源的户型描述
zero_bathroom_layout_counts = zero_bathroom["layout"].value_counts()

zero_bathroom_layout_counts.head(10)
# 户型描述确实明确写了 0卫，但这仍然不代表房源真的没有卫生间。可能原因包括：
# 户型信息没有完整填写
# 卫生间为公共设施
# 爬取时未获得卫生间数量
# 房源并非标准住宅户型
# 现在需要查看标题和面积才能判断。

layout
1室0厅0卫     46
1室1厅0卫     14
1房间0卫       7
2室1厅0卫      6
3室1厅0卫      3
3室0厅0卫      2
未知室0厅0卫     2
3室2厅0卫      2
2房间0卫       1
4室0厅0卫      1
Name: count, dtype: int64

In [27]:
# 查看前10条0卫生间房源
zero_bathroom_sample = zero_bathroom[
    [
        "city",
        "type",
        "bedroom_num",
        "hall_num",
        "bathroom_num",
        "layout",
        "rent_area",
        "rent_price_listing",
        "house_title"
    ]
]

zero_bathroom_sample.head(10)
# 这10条样本说明，bathroom_num = 0 不能当作正常住宅的真实卫生间数量。
# 标题中出现了：写字楼,办公,车位出租,大开间,非标准住宅房源

,city,type,bedroom_num,hall_num,bathroom_num,layout,rent_area,rent_price_listing,house_title
58,北京,整租,1,1,0,1室1厅0卫,80,15000,世纪科贸大厦 1室1厅 15000元
119,北京,整租,1,0,0,1房间0卫,130,39000,"整租 · 朗琴国际写字楼，长期出租,看房方便"
193,北京,整租,1,0,0,1室0厅0卫,121,15500,整租 · 正规朝北120平大开间 办公神器
292,北京,整租,1,0,0,1室0厅0卫,42,1000,整租 · 雅世合金公寓 地下车位出租 随时看房 不能住人
307,北京,整租,1,0,0,1房间0卫,48,2400,整租 · 华远西红世 1房间0卫 2400元
479,北京,整租,1,0,0,1室0厅0卫,12,2300,东四十一条 1室0厅 2300元
506,北京,整租,1,0,0,1室0厅0卫,16,2500,整租 · 北京市第六医院旁边安静温馨小平房
594,北京,整租,1,1,0,1室1厅0卫,257,52500,北环中心 1室1厅 52500元
676,北京,整租,1,0,0,1室0厅0卫,87,14500,整租 · 广渠门方正开间 高楼层视野好 地铁七号线 看房方便
694,北京,整租,2,0,0,2房间0卫,154,13000,整租 · 上奥世纪中心 2房间0卫 13000元


In [28]:
# 检查卫生间数量特别大的房源
large_bathroom = df[df["bathroom_num"] >= 6]

large_bathroom_sample = large_bathroom[
    [
        "city",
        "type",
        "bedroom_num",
        "hall_num",
        "bathroom_num",
        "layout",
        "rent_area",
        "rent_price_listing",
        "house_title"
    ]
]

large_bathroom_sample.head(10)
# 这些高卫生间数量房源基本合理：
# 户型中的卫生间数量与 bathroom_num 一致
# 面积普遍较大，例如342、406、588、949平方米
# 都是整租的大型住宅，可能是别墅或高端大户型

,city,type,bedroom_num,hall_num,bathroom_num,layout,rent_area,rent_price_listing,house_title
1206,北京,整租,9,5,9,9室5厅9卫,544,40000,整租 · 精装修独栋别墅，家具家电齐全，可直接入住！
1779,北京,整租,6,1,7,6室1厅7卫,342,32000,北京华贸城 6室1厅 32000元 跃层
1985,北京,整租,6,2,6,6室2厅6卫,312,90000,整租 · 卓锦万代精装6居室，房子状况新，诚意出租
2385,北京,整租,5,2,6,5室2厅6卫,406,54000,整租 · 燕西台 5室2厅 54000元
3148,上海,整租,4,2,6,4室2厅6卫,197,7500,湖畔天下(别墅) 4室2厅 7500元
3737,上海,整租,5,3,7,5室3厅7卫,588,75000,全明户型 家电齐全，干净温馨 随时可住 近9号线
3778,上海,整租,6,3,8,6室3厅8卫,949,75000,整租 · 性价别墅，可整租可分层出租，价格可谈，看房方便！
3912,上海,整租,5,4,6,5室4厅6卫,436,50000,整租 · 上实滨湖和墅 5室4厅 50000元
4156,上海,整租,4,3,6,4室3厅6卫,473,35000,整租 · 金地天御 4室3厅 35000元
4390,上海,整租,9,4,6,9室4厅6卫,236,34000,整租 · 中山公园顶楼复式精装修，可办公，采光超好，地铁房


## 四、面积与租金格式检查

In [29]:
# 开始检查面积字段，尝试把面积转换为数值
area_numeric = pd.to_numeric(
    df["rent_area"],
    errors="coerce"
)

invalid_area_count = area_numeric.isna().sum()

invalid_area_count
# 共有250条面积记录不能直接转换成普通数字

np.int64(250)

In [30]:
# 查看这些面积的原始格式
invalid_area = df[area_numeric.isna()]

invalid_area_values = invalid_area["rent_area"].value_counts()

invalid_area_values.head(10)
# 面积区间

rent_area
30-35    15
20-25    11
18-20     7
40-45     6
25-28     6
25-30     5
28-33     4
30-40     4
15-18     4
15-20     4
Name: count, dtype: int64

In [31]:
# 检查这些记录是否都包含短横线
area_has_hyphen = invalid_area["rent_area"].str.contains("-")

area_has_hyphen.value_counts()
# 说明250条无法直接转换的面积记录全部都是区间格式，没有发现其他异常文本。

rent_area
True    250
Name: count, dtype: int64

In [32]:
# 检查租金字段有多少条不能直接转换
price_numeric = pd.to_numeric(
    df["rent_price_listing"],
    errors="coerce"
)

invalid_price_count = price_numeric.isna().sum()

invalid_price_count

np.int64(524)

In [33]:
# 查看这些租金的原始格式
invalid_price = df[price_numeric.isna()]

invalid_price_values = invalid_price["rent_price_listing"].value_counts()

invalid_price_values.head(10)

rent_price_listing
1400-1500    7
1200-1300    5
850-900      5
900-1000     5
1300-1400    5
1800-2000    5
1580-1680    4
800-850      4
1000-1100    4
1500-1600    4
Name: count, dtype: int64

In [34]:
# 确认524条记录是否全部包含短横线
price_has_hyphen = invalid_price["rent_price_listing"].str.contains("-")

price_has_hyphen.value_counts()
# 524条无法直接转换的租金记录全部都是区间格式。

rent_price_listing
True    524
Name: count, dtype: int64

In [35]:
# 现在需要写一个基础函数，把普通数字和区间统一转换为数值。我们先只创建并测试函数，不修改数据。
def convert_range(value):
    value = str(value)

    if "-" in value:
        numbers = value.split("-")

        low = float(numbers[0])
        high = float(numbers[1])

        average = (low + high) / 2

        return average

    return float(value)

print(convert_range("30-35"))
print(convert_range("1400"))

32.5
1400.0


In [37]:
# 把转换函数应用到面积和租金字段
area_for_check = df["rent_area"].apply(convert_range) # apply() 表示把convert_range函数依次应用到每一条记录

price_for_check = df["rent_price_listing"].apply(convert_range)

print("面积转换后的缺失数量：", area_for_check.isna().sum())
print("租金转换后的缺失数量：", price_for_check.isna().sum())

面积转换后的缺失数量： 0
租金转换后的缺失数量： 0


In [38]:
# 检查转换后面积的范围
area_for_check.describe()
# 6平方米和1,500平方米与大多数房源差距很大，需要检查原始记录。

count    12000.000000
mean        83.868708
std         60.175978
min          6.000000
25%         46.000000
50%         74.000000
75%        102.000000
max       1500.000000
Name: rent_area, dtype: float64

In [39]:
# 统计面积不超过10平方米的房源数量
small_area = df[area_for_check <= 10]

small_area_count = len(small_area)

small_area_count
# 共有160条房源面积不超过10平方米。这个数量不算极少，可能主要是合租单间。

160

In [40]:
# 检查这些小面积房源的出租类型
small_area_type_counts = small_area["type"].value_counts()

small_area_type_counts
# 148条是合租，面积可能只代表出租房间，基本合理
# 12条是整租，需要重点检查，因为整套房源不超过10平方米比较少见

type
合租    148
整租     12
Name: count, dtype: int64

In [41]:
# 查看这12条小面积整租房源
small_entire = small_area[small_area["type"] == "整租"]

small_entire[
    [
        "city",
        "rent_area",
        "layout",
        "rent_price_listing",
        "house_title"
    ]
]
# 这12条小面积整租存在多种情况，不能统一删除：
# 6平方米、3室0厅0卫：面积与户型明显不一致
# 10平方米、3室2厅2卫：标题明确是车位，属于非住宅数据
# 标题包含“两房合租”，但 type 写成整租：出租类型可能标错
# 部分9至10平方米记录可能是真实的小单间或公寓

,city,rent_area,layout,rent_price_listing,house_title
1824,北京,6,3室0厅0卫,1800,周仓庵胡同 3室0厅 1800元
4386,上海,7,1室0厅0卫,6000,整租 · 泰安路20号 1室0厅 6000元
4695,上海,10,3室2厅2卫,1300,只出租一个产权车位！！！！！！！！！！！
6042,广州,6-8,2室1厅1卫,2680-2760,世联红璞公寓 穗和家园店-H2栋 温馨两房一厅 二居+
6553,广州,10,1室0厅1卫,1600,e寓共享公寓 E寓流花站前店 精致复式LOFT 开间
7905,广州,10,1室0厅1卫,450,优居乐公寓 双岗店A栋 舒适单间 开间
9855,深圳,10,1室1厅1卫,1356,趣客公寓 流塘店 小LOFT单房
9962,深圳,9,1室0厅1卫,1380,粤鹏湾公馆 吉华店 小单间 开间
10854,深圳,10,1室2厅1卫,2800,欧式简约风格园林小区、停车位充足、品质住宅！！
10884,深圳,9,1室0厅1卫,1700,科苑西住宅小区单间出租，步行科兴科学园


In [42]:
# 统计超大面积房源
large_area = df[area_for_check >= 500]

large_area_count = len(large_area)

large_area_count
# 共有14条房源面积达到或超过500平方米，数量很少，需要检查是否为别墅、商业房源或面积录入错误。

14

In [43]:
# 查看这14条超大面积房源
large_area[
    [
        "city",
        "type",
        "rent_area",
        "layout",
        "rent_price_listing",
        "house_title"
    ]
]
# 明显可疑的记录：
# 1室1厅1卫
# 面积1,500平方米
# 月租仅1,500元
# 这条记录的面积、户型和租金明显冲突，很可能是面积录入错误。另外标题含“仓库”的房源属于非标准住宅。

,city,type,rent_area,layout,rent_price_listing,house_title
1206,北京,整租,544,9室5厅9卫,40000,整租 · 精装修独栋别墅，家具家电齐全，可直接入住！
1671,北京,整租,515,5室3厅4卫,26000,整租 · 麦卡伦地 5室3厅 26000元
3043,上海,整租,518,5室2厅4卫,25000,紫都上海晶园 5室2厅 25000元
3045,上海,整租,510,5室2厅3卫,10000,新南路壹号 完全人车分流小区 大独栋 可做仓库
3621,上海,整租,556,4室2厅3卫,15000,长泰西郊 隶属洞泾 近松江漕河泾开发区 拉菲云廊
3737,上海,整租,588,5室3厅7卫,75000,全明户型 家电齐全，干净温馨 随时可住 近9号线
3778,上海,整租,949,6室3厅8卫,75000,整租 · 性价别墅，可整租可分层出租，价格可谈，看房方便！
4808,上海,整租,531,4室3厅3卫,45000,仁恒精致四房，定制装修，保养如新，诚意出租欢迎来电
4974,上海,整租,530,5室3厅4卫,75000,小区高区5房，有车位 包物业包发票、大面积
5005,上海,整租,515,5室3厅4卫,30000,长堤花园独栋，自住精装修，中、央空调地暖，大花园


In [44]:
# 检查转换后租金的整体范围
price_for_check.describe()
# 平均数明显高于中位数，说明少量高价房源把平均值拉高了。300元和180,000元都需要检查。

count     12000.000000
mean       7103.014167
std        8360.216602
min         300.000000
25%        2923.750000
50%        4800.000000
75%        7800.000000
max      180000.000000
Name: rent_price_listing, dtype: float64

In [45]:
# 统计月租不超过500元的房源数量
low_price = df[price_for_check <= 500]

low_price_count = len(low_price)

low_price_count
# 共有28条房源月租不超过500元，数量很少，需要检查它们是低价合租、车位、价格单位问题，还是错误数据

28

In [46]:
# 检查这些低价房源的出租类型
low_price_type_counts = low_price["type"].value_counts()

low_price_type_counts

type
整租    28
Name: count, dtype: int64

In [47]:
# 查看前10条低价房源
low_price_sample = low_price[
    [
        "city",
        "rent_area",
        "layout",
        "rent_price_listing",
        "rent_price_unit",
        "house_title"
    ]
]

low_price_sample.head(10)

,city,rent_area,layout,rent_price_listing,rent_price_unit,house_title
6002,广州,35,1室1厅1卫,480,元/月,职业房东 小居公寓 一房一厅30501
6358,广州,85,2室2厅1卫,500,元/月,镇安横二巷 2室2厅 500元
6611,广州,30,1室1厅1卫,380-450,元/月,职业房东 小居公寓 特价一房一厅
6755,广州,20,1室0厅1卫,500,元/月,尔必公寓 庙头和乐里3号 f401单间 开间
7075,广州,24,1室0厅1卫,450,元/月,职业房东 爱特公寓珠村店 单间15-502 开间
7094,广州,12,1室0厅0卫,300,元/月,万科东荟城 1室0厅 300元
7115,广州,20,1室0厅1卫,500,元/月,旗寓 小洲村4店 ❤单间❤小洲4栋 开间
7133,广州,20,1室0厅1卫,450,元/月,尔必公寓 东湾街八巷10号 23106单间 开间
7165,广州,20,1室1厅1卫,500,元/月,东塱新爵村 单间出租
7579,广州,20,1室0厅1卫,450,元/月,尔必公寓 东湾街八巷10号 23403单间 开间


In [48]:
# 统计月租达到100,000元及以上的房源
high_price = df[price_for_check >= 100000]

high_price_count = len(high_price)

high_price_count

7

In [49]:
# 查看这7条房源
high_price[
    [
        "city",
        "type",
        "rent_area",
        "layout",
        "rent_price_listing",
        "house_title"
    ]
]
# 基本合理

,city,type,rent_area,layout,rent_price_listing,house_title
2617,北京,整租,317,5室3厅5卫,150000,整租 · 棕榈泉国际公寓 5室3厅 150000元
3566,上海,整租,345,4室2厅4卫,100000,滨江凯旋门小区10万，业主诚意出租，江景好！
4566,上海,整租,378,4室2厅4卫,120000,凯德茂名公馆 4室2厅 120000元
4996,上海,整租,426,5室4厅5卫,180000,紫藤居(别墅) 5室4厅 180000元
9062,深圳,整租,331,3室2厅4卫,160000,深圳湾一号 奢华3+1房，360度一线海景.
9841,深圳,整租,292,4室2厅3卫,160000,深圳湾1号 4室2厅 160000元
10648,深圳,整租,275,8室2厅7卫,120000,三湘海尚花园 8室2厅 120000元


## 五、缺失值与地理字段检查

In [50]:
# 显示存在缺失值的字段
missing_count = df.isna().sum()

missing_count = missing_count[missing_count > 0]

missing_count

bizcircle_name          1
distance             5206
frame_orientation     101
house_tag            1876
latitude               31
longitude              31
resblock_name        1514
dtype: int64

In [51]:
# 计算缺失比例
missing_rate = missing_count / len(df) * 100

missing_rate = missing_rate.round(2)

missing_rate

bizcircle_name        0.01
distance             43.38
frame_orientation     0.84
house_tag            15.63
latitude              0.26
longitude             0.26
resblock_name        12.62
dtype: float64

In [52]:
# 检查地铁距离缺失主要来自哪些城市
missing_distance = df[df["distance"].isna()]

missing_distance_city_counts = missing_distance["city"].value_counts()

missing_distance_city_counts
# 说明缺失并不是四个城市均匀发生的，广州和深圳更严重

city
广州    1583
深圳    1579
上海    1056
北京     988
Name: count, dtype: int64

In [53]:
# 检查经纬度缺失是否发生在同一批房源
latitude_missing = df["latitude"].isna()

longitude_missing = df["longitude"].isna()

missing_coordinate = df[latitude_missing | longitude_missing]

len(missing_coordinate)
# 说明经纬度缺失集中在同一批31条房源中；没有出现大量“只有经度、没有纬度”的额外记录。

31

In [54]:
# 检查这31条房源来自哪些城市
missing_coordinate_city_counts = missing_coordinate["city"].value_counts()

missing_coordinate_city_counts

city
广州    13
上海     8
深圳     6
北京     4
Name: count, dtype: int64

In [55]:
# 检查经纬度和地铁距离的数值范围
geo_columns = ["latitude", "longitude", "distance"]

df[geo_columns].describe()
# 基本合理

,latitude,longitude,distance
count,11969.000000,11969.000000,6794.000000
mean,29.218101,116.311657,549.947012
std,7.065259,3.175260,317.187338
min,22.330000,113.170910,1.000000
25%,22.786416,113.810439,295.000000
50%,30.827435,116.077893,525.000000
75%,39.676703,116.722221,796.000000
max,40.235529,121.923456,1199.000000


## 数据质量深入分析总结

### 数据基本情况

- 数据包含12,000条租房记录和20个字段。
- 北京、上海、广州、深圳各有3,000条记录。
- 整租房源11,586条，合租房源414条。
- 所有租金单位均为“元/月”。
- 房源ID没有重复，每条记录代表一条独立房源。
- 已建立包含字段含义、数据类型、缺失数量、缺失比例、唯一值数量和示例值的数据字典。

### 分类字段检查

- 数据包含4个城市和49个行政区。
- `layout`、`frame_orientation`和`house_tag`存在大量组合取值。
- 房屋朝向和标签字段需要在实际分析时再进行拆分或标准化。
- 部分房源标题与`type`字段存在不一致，例如标题写“合租”，但字段标记为“整租”。

### 房间数量检查

- `bedroom_num`为0的记录有3条，户型显示“未知室”，应改为缺失值。
- 卧室数达到10个及以上的记录有4条，多为大型合租房或别墅，可以保留。
- `hall_num`为0的记录有1,502条，其中大多数是“1室0厅1卫”等合理户型。
- `hall_num`最大为5，对应大型住宅，属于合理值。
- `bathroom_num`为0的记录有87条，部分属于写字楼、车位或非标准住宅。
- 卫生间数量较大的记录与大面积户型基本一致，可以保留。

### 面积字段检查

- 250条面积记录使用区间格式，例如`30-35`。
- 所有无法直接转换的面积记录都包含短横线，可以统一取区间中点。
- 转换后面积中位数为74平方米，最小值为6平方米，最大值为1,500平方米。
- 面积不超过10平方米的房源有160条，其中148条为合租，12条为整租。
- 小面积合租房源通常表示出租房间面积，可以保留。
- 小面积整租中存在车位、类型标记错误和面积与户型不一致的问题。
- 面积达到500平方米及以上的房源有14条，多数为别墅。
- 1室1厅、1,500平方米、月租1,500元的记录明显异常。

### 租金字段检查

- 524条租金记录使用区间格式，例如`1400-1500`。
- 所有无法直接转换的租金记录都包含短横线，可以统一取区间中点。
- 转换后租金中位数为4,800元/月。
- 最低租金为300元/月，最高租金为180,000元/月。
- 月租不超过500元的记录有28条，并且全部标记为整租。
- 低价记录的标题、面积、出租类型和租金字段存在明显不一致。
- 月租达到100,000元及以上的记录有7条，均为大型高端住宅，基本合理。

### 缺失值检查

- `distance`缺失5,206条，缺失比例为43.38%。
- 广州和深圳的地铁距离缺失比例高于北京和上海。
- `house_tag`缺失1,876条。
- `resblock_name`缺失1,514条。
- `frame_orientation`缺失101条。
- 经纬度同时缺失31条。
- `bizcircle_name`缺失1条。

### 第3天清洗规则

1. 不修改原始CSV，清洗结果保存到`data/processed`。
2. 保留原始面积和租金字段，新增清洗后的数值字段。
3. 面积和租金区间统一取区间中点。
4. `bedroom_num`为0的记录改为缺失值。
5. `hall_num`为0属于合理值，继续保留。
6. 明确属于写字楼、办公室、车位、仓库等非住宅房源的记录排除。
7. `bathroom_num`为0且无法确认真实数量的记录改为缺失值。
8. 小面积整租、超大面积异常和低价整租记录添加异常标记。
9. 高卧室数、高客厅数、高卫生间数和高租金不能只凭数值删除，需要结合户型、面积和标题判断。
10. `distance`缺失不填平均值，只在距离分析时排除。
11. 经纬度缺失不进行人工填补，只在地图分析时排除。
12. `house_tag`缺失填充为“无标签”。
13. `resblock_name`、`frame_orientation`和`bizcircle_name`缺失时分别填充为“未知小区”“未知朝向”和“未知商圈”。
14. 整租和合租需要分开分析，避免面积和户型含义不同造成误导。